# Causal Uplift Modeling on Criteo v2.1
## Research Problem, Design, and Experimental Roadmap

**Purpose.** This is the front door to the research project. It summarizes the frozen research question, causal design, estimator roles, evaluation protocol, and modular workflow. It performs no data audit, model training, model selection, or held-out evaluation.

**Authority.** The owner-approved `docs/decision_register.csv` controls project decisions. The numbered specifications in `docs/01_causal_contract.md` through `docs/07_metric_specification.md` control their named contracts.


## 1. Research motivation

Traditional response modeling asks **who is likely to convert**, approximately targeting

$$P(Y=1\mid X).$$

Uplift or heterogeneous-treatment-effect modeling asks **whose outcome changes because treatment is assigned**, targeting

$$\tau(x)=E[Y(1)-Y(0)\mid X=x].$$

A high conversion probability does not imply a high incremental treatment effect. The response model is therefore a useful targeting reference, but not an uplift estimator.


## 2. Research objective and questions

**Frozen framing (D10): comparative uplift/CATE evaluation.** The objective is to evaluate assignment-effect ranking strategies at Criteo scale under one common, leakage-controlled protocol—not to declare a winner in advance.

**RQ1.** How do response targeting, T-Learner, and cross-fitted X-Learner differ in out-of-sample uplift ranking under the same population, split, and metric conventions?

**RQ2.** How does X-Learner behave relative to T-Learner under the empirically strong treatment/control allocation imbalance, without assuming in advance that imbalance guarantees an advantage?

**RQ3.** How do the accepted meta-learner comparators and Causal Forest compare in uplift-ranking performance, training-seed stability, and computational cost after their implementation gates pass?

DR-Learner is a conditional stretch comparator. S-Learner is **DEFERRED** and is not part of the frozen active comparison unless a later owner-approved decision promotes it.


## 3. Dataset and causal setup

| Item | Frozen role |
|---|---|
| Dataset | CRITEO-UPLIFTv2.1; current canonical artifact contains 13,979,592 observed rows |
| Unit | One released row, not an assumed unique user |
| Covariates `X` | Exactly ordered `f0` through `f11` |
| Treatment `T` | Binary randomized assignment indicator `treatment` |
| Primary outcome `Y` | Binary `conversion` |
| Secondary outcome | `visit`, in a separate robustness pipeline |
| Audit-only field | `exposure`, treated conservatively as post-assignment |
| Primary estimand | Assignment/intention-to-treat CATE for ranking: $\tau(x)=E[Y(1)-Y(0)\mid X=x]$ |

`exposure` is not the primary treatment, is excluded from `X`, and cannot filter the primary population. Empirical balance and exposure checks may support or limit interpretation, but they do not prove randomization or causal identification.


## 4. Frozen experimental contract

| Contract item | Frozen decision | Source |
|---|---|---|
| Estimand | Assignment/ITT CATE ranking; assigned-arm ATE is an aggregate summary | D01; docs/01 |
| Features | `f0`–`f11` only; labels, treatment, visit, exposure and identity are forbidden | D05; docs/02 |
| Treatment | `treatment`; `exposure` cannot replace it | D01; docs/01 |
| Outcomes | `conversion` primary; `visit` separate secondary | D02; docs/01–02 |
| Population | All released rows passing predeclared hard integrity gates | docs/01 |
| Precision | `float64` primary; `float32` sensitivity only | D09; docs/02 |
| Split | Frozen 70/15/15 train/validation/test, joint `(T,Y)` stratification, seed 42 | D06; docs/06 |
| Duplicate policy | Keep all eligible rows; deduplication/profile removal is sensitivity-only | D07–D08; docs/04 |
| Leakage policy | Validation selects; held-out test is used once only after pre-test freeze | docs/06 |
| Primary metric | Qini above the theoretical expected-random line | D24; docs/07 |
| Secondary metrics | Raw Qini area; fixed `uplift@K`; incremental conversions | D25–D26; docs/07 |
| Uncertainty | 500 paired arm-stratified bootstrap draws plus complete seed reporting | D29; docs/06–07 |

The D23 sequence `50K → 500K → 2M → full` is an implementation/resource promotion ladder. It is not a 2M-row cap on the primary audit or final held-out evaluation.


## 5. Modeling strategy

| Method | Research role | Frozen status |
|---|---|---|
| Theoretical/seeded random ranking | Sanity and expected-random reference | Required |
| Response LightGBM | Predictive targeting reference: $P(Y=1\mid X)$; not causal | Required |
| T-Learner | Two LightGBM factual-outcome surfaces, $\hat\mu_1(X)-\hat\mu_0(X)$ | Required primary causal baseline |
| X-Learner | Cross-fitted signed pseudo-effects; tests whether its architecture helps under arm imbalance | Accepted main comparator |
| Causal Forest | Direct heterogeneous-effect forest comparator | Accepted main comparator; exact implementation provisional |
| DR-Learner | Orthogonalized pseudo-outcome comparator | Conditional stretch; promotion-gated |
| S-Learner | Single outcome model evaluated at `T=1` and `T=0` | Deferred |

LightGBM is the selected scalable base-learner family for authorized response/outcome/effect stages. Exact packages, hyperparameters, iteration counts, serialization, and scale evidence remain Sprint 2 work. No method is declared the winner in advance.


## 6. Evaluation strategy

All frozen methods are compared on exactly the same evaluation rows, ranked by descending score with `_source_row_id` as the deterministic tie-breaker.

- **Primary:** `qini_above_random` under the exact unnormalized raw-Qini convention in `docs/07_metric_specification.md`.
- **Secondary:** raw Qini area, fixed `uplift@{10%,20%,30%,50%,100%}`, and estimated incremental conversions.
- **Response diagnostics only:** ROC-AUC, average precision, and log loss do not measure uplift-ranking quality.
- **Test-sample uncertainty:** 500 paired treatment-arm-stratified bootstrap draws on frozen predictions.
- **Training stability:** complete results for seeds `42`, `123`, and `2026`; no favorable-seed selection.

Only one potential outcome is observed for each real row. The project therefore does not report empirical PEHE or direct true-ITE accuracy on CRITEO-UPLIFTv2.1. Those quantities require an explicitly synthetic or semi-synthetic experiment with known ground truth.


## 7. Modular workflow

```text
00 Project overview
  ↓
01 Full-data integrity and feasibility evidence
  ↓
02 Response + T baseline development
     (S-Learner remains deferred unless separately approved)
  ↓
03 X-Learner
  ↓
04 Causal Forest
  ↓
05 Evaluation and robustness
  ↓
99 Final research report (later)
```

Modeling notebooks will persist versioned configurations, manifests, models, predictions, and metric tables. The future final report will synthesize frozen artifacts rather than retraining the project.


## 8. Current status and next boundary

| Area | Current evidence-based status |
|---|---|
| Sprint 1 design | `FROZEN_WITH_DOCUMENTED_LIMITATIONS` |
| Full-data structural audit | Implemented in `01_sprint1_feasibility_evidence.ipynb` for all 13,979,592 rows |
| Canonical source integrity | Official local `.csv.gz` is hashed and reconciled to the working CSV by Notebook 01 |
| Model implementation | Not started in this notebook architecture |
| Pre-test executable freeze | Not yet complete |
| Held-out evaluation | Not accessed or reported |

**Next boundary:** complete CSV-to-Parquet lineage, executable data/split manifests, and the applicable D23 correctness/resource gate, then begin response and T-Learner development using validation-only feedback. The canonical compressed-source checksum is externally reconciled; field-timing evidence, the exact assignment mechanism, exact model configurations, and provisional promotion evidence remain TBD where the repository has not frozen them.
